<a href="https://colab.research.google.com/github/Sunbean24/Motorek-PP-projekt/blob/main/rowerek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/dynamicslab/pysindy.git

  Cloning https://github.com/dynamicslab/pysindy.git to /tmp/pip-req-build-j7hxbtjj
  Running command git clone --filter=blob:none --quiet https://github.com/dynamicslab/pysindy.git /tmp/pip-req-build-j7hxbtjj
  Resolved https://github.com/dynamicslab/pysindy.git to commit c4421fcec275c8f4cc5c1e93bebb961b212067ae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pysindy: filename=pysindy-2.1.1.dev111+gc4421fcec-py3-none-any.whl size=150637 sha256=3dede55f81ee1699c19218d25d3872cbf515179c1a53b9491b3d4e672af1750f
  Stored in directory: /tmp/pip-ephem-wheel-cache-dnea8ofd/wheels/b7/0d/0e/6b0cc09c0e17f0e7035d8abd84a339c8d6af9112fc6e986259
Successfully built pysindy


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.signal import savgol_filter
import warnings

import pysindy as ps

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. ARCHITECTURES (SIMPLIFIED) ---
import torch
import torch.nn as nn

class MambaLikeSSM(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, output_size=4):
        super().__init__()

        self.in_proj = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True
        )

        self.out_proj = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: [Batch, SeqLen, 5]

        # Input Expansion
        x_emb = self.activation(self.in_proj(x))

        # Sequence Mixing (discarding the hidden state tuple)
        gru_out, _ = self.gru(x_emb)

        # Project to physical states
        out = self.out_proj(gru_out)

        return out


Using device: cuda


In [ ]:
# --- 2. GLOBAL PHYSICS ENGINE & DATA GENERATION / LOADING ---
import pandas as pd
import numpy as np
import torch
import glob
from scipy.signal import sawtooth

# =====================================================================
# GLOBALNE PARAMETRY SPRZĘTOWE I FIZYCZNE
# =====================================================================
USE_REAL_DATA = True       # Przełącznik: True dla Twoich CSV, False dla symulacji
CSV_PATH_PATTERN = "dane/*.csv" # Ścieżka do Twoich plików

# Parametry enkodera i sterowania
PPR = 48.0                 # Rozdzielczość enkodera (impulsy na obrót)
PWM_MAX = 255.0            # Maksymalna wartość PWM
V_MAX = 12.0               # Maksymalne napięcie zasilania (V)

# Parametry fizyczne wahadła (wykorzystywane m.in. w symulacji i EKF)
GRAVITY = 9.81             # Przyspieszenie ziemskie [m/s^2]
PENDULUM_LENGTH = 0.03     # Długość wahadła do środka masy [m]
# =====================================================================

def load_real_data(file_pattern, ppr, pwm_max, v_max, chunk_size=200):
    noisy_seqs, control_seqs = [], []
    file_paths = glob.glob(file_pattern)

    if not file_paths:
        print(f"Brak plików w lokalizacji: {file_pattern}")
        return None, None, 0.01

    print(f"Wczytywanie {len(file_paths)} plików z rzeczywistymi pomiarami...")

    dt_seq = 0.01
    odrzucone_pliki = 0

    for path in file_paths:
        df = pd.read_csv(path)

        if len(df) < chunk_size:
            odrzucone_pliki += 1
            continue

        # Pobieranie danych (ignorujemy timestampy do obliczeń fizycznych)
        theta = df['angle'].values
        gyro = -df['velocity'].values
        accel = df['acceleration'].values
        pwm = df['pwm'].values

        # Obliczanie prędkości koła na podstawie idealnego, sztywnego czasu dt = 0.01
        wheel_angle = df['wheel_angle'].values
        delta_angle = np.diff(wheel_angle, prepend=wheel_angle[0])
        omega_w = delta_angle / dt_seq

        u = (pwm / pwm_max) * v_max

        noisy = np.column_stack((theta, gyro, accel, omega_w))

        num_chunks = len(noisy) // chunk_size

        for i in range(num_chunks):
            start_idx = i * chunk_size
            end_idx = start_idx + chunk_size

            noisy_seqs.append(noisy[start_idx:end_idx])
            control_seqs.append(u[start_idx:end_idx])

    print(f"Wygenerowano {len(noisy_seqs)} równych sekwencji o długości {chunk_size} próbek.")
    if odrzucone_pliki > 0:
        print(f"Odrzucono {odrzucone_pliki} plików (były krótsze niż {chunk_size} próbek).")

    return np.array(noisy_seqs, dtype=np.float32), np.array(control_seqs, dtype=np.float32), dt_seq
def generate_reaction_wheel_pendulum(num_seqs=20, steps=5000, dt=0.02):
    np.random.seed(42)
    m_p, l, g = 0.15, PENDULUM_LENGTH, GRAVITY
    I_p, I_w = 0.003, 0.0001
    b_p, b_w = 0.001, 0.0005
    K_t, K_e = 0.05, 0.05
    u_d = 1.5

    def physics_step(theta, omega_p, omega_w, u):
        u_net = u - K_e * omega_w
        u_eff = u_net - u_d * np.tanh(u_net / u_d)
        tau_m = K_t * u_eff
        alpha_p = (m_p * g * l * np.sin(theta) - tau_m + b_w * omega_w - b_p * omega_p) / I_p
        alpha_w_new = (tau_m - b_w * omega_w) / I_w - alpha_p
        next_omega_p = omega_p + alpha_p * dt
        next_theta = theta + next_omega_p * dt
        next_omega_w = omega_w + alpha_w_new * dt
        return next_theta, next_omega_p, next_omega_w, alpha_p, alpha_w_new

    clean_seqs, noisy_seqs, control_seqs = [], [], []
    t_sim = np.arange(0, steps * dt, dt)
    enc_res = (2 * np.pi) / PPR

    for i in range(num_seqs):
        mode = i % 4
        if mode == 0:
            f = np.random.uniform(0.1, 0.3)
            u = np.random.uniform(7.0, 11.0) * np.sin(2 * np.pi * f * t_sim)
        elif mode == 1:
            f = np.random.uniform(0.1, 0.4)
            u = np.random.uniform(6.0, 10.0) * sawtooth(2 * np.pi * f * t_sim, width=0.5)
        elif mode == 2:
            f0, f1 = 0.05, 0.6
            u = np.random.uniform(5.0, 9.0) * np.sin(2 * np.pi * (f0 + (f1 - f0) * t_sim / (steps*dt)) * t_sim)
        else:
            u = 5.0 * np.sin(0.4 * t_sim) + 3.0 * np.sin(0.9 * t_sim + 1.0) + 2.0 * np.sin(0.15 * t_sim)

        theta = np.random.uniform(-np.pi, np.pi)
        omega_p, phi, omega_w = np.random.uniform(-1.0, 1.0), 0.0, 0.0
        clean, noisy, phi_enc_prev = [], [], 0.0

        for k in range(steps):
            tau_m_sens = K_t * (u[k] - K_e * omega_w - u_d * np.tanh((u[k] - K_e * omega_w) / u_d))
            curr_alpha_p = (m_p * g * l * np.sin(theta) - tau_m_sens + b_w * omega_w - b_p * omega_p) / I_p
            sens_gyro = omega_p + np.random.normal(0, 0.02)
            sens_accel = g * np.sin(theta) + l * curr_alpha_p + np.random.normal(0, 0.1)
            sens_theta = theta + np.random.normal(0, 0.015)

            phi_enc = np.round(phi / enc_res) * enc_res
            sens_omega_w = (phi_enc - phi_enc_prev) / dt
            phi_enc_prev = phi_enc

            clean.append([theta, omega_p, omega_w])
            noisy.append([sens_theta, sens_gyro, sens_accel, sens_omega_w])
            theta, omega_p, omega_w, _, _ = physics_step(theta, omega_p, omega_w, u[k])
            phi += omega_w * dt

        clean_seqs.append(clean)
        noisy_seqs.append(noisy)
        control_seqs.append(u)

    return np.array(clean_seqs, dtype=np.float32), np.array(noisy_seqs, dtype=np.float32), np.array(control_seqs, dtype=np.float32)

# --- Ładowanie odpowiednich danych ---
if USE_REAL_DATA:
    noisy_all, u_all, dt = load_real_data(CSV_PATH_PATTERN, PPR, PWM_MAX, V_MAX)
    # Zastępcza "prawda" dla rzeczywistych danych (uzupełniana w klasycznym potoku)
    clean_all = np.zeros_like(noisy_all[:, :, :3])
    num_train = int(len(noisy_all) * 0.8)
else:
    clean_all, noisy_all, u_all = generate_reaction_wheel_pendulum(num_seqs=20)
    dt = 0.02
    num_train = 16

X_train_np, U_train_np = noisy_all[:num_train], u_all[:num_train]
X_val_np, U_val_np = noisy_all[num_train:], u_all[num_train:]

Train_Input = torch.from_numpy(np.concatenate([X_train_np, U_train_np[:, :, None]], axis=-1)).to(device).float()
Val_Input = torch.from_numpy(np.concatenate([X_val_np, U_val_np[:, :, None]], axis=-1)).to(device).float()

print(f"Block 2 Ready: Zbudowano zbiór. Trening: {len(X_train_np)} sekwencji, Walidacja: {len(X_val_np)} sekwencji. Krok czasowy dt = {dt:.4f}s")

Brak plików w lokalizacji: dane/*.csv


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# --- 3. SINDy (BINDy) IDENTIFICATION & COMPILED EKF ---
import pysindy as ps
import numpy as np
import torch
from scipy.signal import savgol_filter
from numba import njit

print("Smoothing Sensor Data for SINDy Training...")
x_train_sindy, u_train_sindy, t_train_sindy = [], [], []

for i in range(num_train):
    meas = X_train_np[i]
    th_raw = meas[:, 0]
    gyro_raw = meas[:, 1]
    om_w_raw = meas[:, 3]

    th_smooth = savgol_filter(th_raw, window_length=15, polyorder=2)
    gyro_smooth = savgol_filter(gyro_raw, window_length=15, polyorder=2)
    om_w_smooth = savgol_filter(om_w_raw, window_length=15, polyorder=2)

    u_train = U_train_np[i].reshape(-1, 1)
    t_train = np.arange(len(th_raw)) * dt

    sub = 3
    x_train_sindy.append(np.stack([th_smooth[::sub], gyro_smooth[::sub], om_w_smooth[::sub]], axis=-1))
    u_train_sindy.append(u_train[::sub])
    t_train_sindy.append(t_train[::sub])

print("Running BINDy with Multi-Tanh Grey-Box Library...")

lib_linear = ps.PolynomialLibrary(degree=1, include_bias=False)
lib_sin = ps.CustomLibrary(library_functions=[lambda x: np.sin(x)], function_names=[lambda x: f"sin({x})"])

lib_tanh = ps.CustomLibrary(
    library_functions=[
        lambda x: np.tanh(0.5 * x),
        lambda x: np.tanh(1.0 * x),
        lambda x: np.tanh(2.0 * x),
        lambda x: np.tanh(5.0 * x)
    ],
    function_names=[
        lambda x: f"tanh(0.5*{x})",
        lambda x: f"tanh({x})",
        lambda x: f"tanh(2.0*{x})",
        lambda x: f"tanh(5.0*{x})"
    ]
)

targeted_library = ps.GeneralizedLibrary(
    [lib_linear, lib_sin, lib_tanh],
    inputs_per_library=[[1, 2, 3], [0], [3]]
)

optimiser = ps.STLSQ(threshold=0.05, alpha=0.5)

sindy_model = ps.BINDy(
    sigma_x=0.2,
    feature_library=targeted_library,
    optimizer=optimiser,
    differentiation_method=ps.FiniteDifference(drop_endpoints=True)
)
sindy_model.fit(x_train_sindy, u=u_train_sindy, t=t_train_sindy)

features = sindy_model.get_feature_names()
x1_idx = features.index('x1')
sindy_model.optimizer.coef_[0, :] = 0.0
sindy_model.optimizer.coef_[0, x1_idx] = 1.0

print("\n--- Discovered SINDy Equations ---")
sindy_model.print()

print("\nCompiling Generic SINDy-EKF...")

numpy_features = []
for f in features:
    f = f.replace('x0', 'x[0]').replace('x1', 'x[1]').replace('x2', 'x[2]')
    f = f.replace('u0', 'u').replace('sin', 'np.sin')
    f = f.replace('tanh(0.5*u)', 'np.tanh(0.5 * u)')
    f = f.replace('tanh(u)', 'np.tanh(u)')
    f = f.replace('tanh(2.0*u)', 'np.tanh(2.0 * u)')
    f = f.replace('tanh(5.0*u)', 'np.tanh(5.0 * u)')
    numpy_features.append(f)

# WSTRZYKIWANIE PARAMETRÓW FIZYCZNYCH ZAMIAST HARDKODOWANIA
numba_code_str = f"""
import numpy as np
from numba import njit

@njit
def _fast_physics_compiled(x, u, C):
    features = np.array([{', '.join(numpy_features)}], dtype=np.float64)
    return C @ features

@njit
def _run_compiled_ekf(z_meas, u_ctrl, C, dt):
    N = len(z_meas)
    x_hat = np.zeros((3, N), dtype=np.float64)
    P_cov = np.eye(3, dtype=np.float64) * 0.1

    Q = np.array([[1e-5, 0.0, 0.0], [0.0, 1e-3, 0.0], [0.0, 0.0, 1e-3]], dtype=np.float64)

    R = np.array([
        [1e-2, 0.0,  0.0,  0.0 ],
        [0.0,  1e-2, 0.0,  0.0 ],
        [0.0,  0.0,  1e-1, 0.0 ],
        [0.0,  0.0,  0.0,  2.0 ]
    ], dtype=np.float64)

    x_hat[:, 0] = np.array([z_meas[0, 0], z_meas[0, 1], z_meas[0, 3]], dtype=np.float64)

    for k in range(1, N):
        curr_x = x_hat[:, k-1]
        curr_u = u_ctrl[k-1]

        x_dot = _fast_physics_compiled(curr_x, curr_u, C)
        x_pred = curr_x + x_dot * dt

        eps = 1e-4
        F_mat = np.zeros((3, 3), dtype=np.float64)
        for i in range(3):
            x_plus = curr_x.copy()
            x_plus[i] += eps
            dot_plus = _fast_physics_compiled(x_plus, curr_u, C)
            F_mat[:, i] = (dot_plus - x_dot) / eps

        F_mat = np.eye(3, dtype=np.float64) + F_mat * dt
        P_pred = F_mat @ P_cov @ F_mat.T + Q

        z_pred = np.zeros(4, dtype=np.float64)
        x_dot_pred = _fast_physics_compiled(x_pred, curr_u, C)
        z_pred[0] = x_pred[0]
        z_pred[1] = x_pred[1]

        # Wykorzystanie wstrzykniętych parametrów
        z_pred[2] = {GRAVITY} * np.sin(x_pred[0]) + {PENDULUM_LENGTH} * x_dot_pred[1]
        z_pred[3] = x_pred[2]

        H_dyn = np.zeros((4, 3), dtype=np.float64)
        for i in range(3):
            x_plus = x_pred.copy()
            x_plus[i] += eps
            z_plus = np.zeros(4, dtype=np.float64)
            x_dot_plus = _fast_physics_compiled(x_plus, curr_u, C)
            z_plus[0] = x_plus[0]
            z_plus[1] = x_plus[1]

            # Wykorzystanie wstrzykniętych parametrów
            z_plus[2] = {GRAVITY} * np.sin(x_plus[0]) + {PENDULUM_LENGTH} * x_dot_plus[1]
            z_plus[3] = x_plus[2]
            H_dyn[:, i] = (z_plus - z_pred) / eps

        S = H_dyn @ P_pred @ H_dyn.T + R
        K = P_pred @ H_dyn.T @ np.linalg.inv(S)

        y = z_meas[k] - z_pred
        y[0] = (y[0] + np.pi) % (2.0 * np.pi) - np.pi

        x_hat[:, k] = x_pred + K @ y
        P_cov = (np.eye(3, dtype=np.float64) - K @ H_dyn) @ P_pred

    out_states = np.zeros((N, 4), dtype=np.float64)
    for k in range(N):
        curr_x = x_hat[:, k]
        curr_u = u_ctrl[k]
        x_dot = _fast_physics_compiled(curr_x, curr_u, C)

        out_states[k, 0] = curr_x[0]
        out_states[k, 1] = curr_x[1]
        out_states[k, 2] = curr_x[2]
        out_states[k, 3] = x_dot[2]

    return out_states
"""
exec_globals = {'np': np, 'njit': njit}
exec(numba_code_str, exec_globals)
_run_compiled_ekf = exec_globals['_run_compiled_ekf']
C_matrix = sindy_model.coefficients().astype(np.float64)

def run_fast_ekf(z_meas, u_ctrl, dt_val=dt):
    return _run_compiled_ekf(z_meas.astype(np.float64), u_ctrl.astype(np.float64), C_matrix, dt_val)

print("Running EKF over all sequences...")
EKF_Train_Labels = torch.from_numpy(np.array([run_fast_ekf(X_train_np[i], U_train_np[i]) for i in range(num_train)])).to(device).float()
EKF_Val_Labels = torch.from_numpy(np.array([run_fast_ekf(X_val_np[i], U_val_np[i]) for i in range(len(X_val_np))])).to(device).float()
print("BINDy-EKF Identification Complete!")

In [ ]:
# --- 4. TRAINING MAMBA APPRENTICES (EKF vs CLASSICAL) ---
from scipy.signal import butter, filtfilt, savgol_filter
from scipy.ndimage import median_filter
import copy
import torch
import torch.nn as nn
import torch.optim as optim

print("Generating Classical Filtering Targets (Median -> FiltFilt -> SavGol -> Deriv)...")

def apply_classical_pipeline(noisy_seq, dt_val=0.02):
    filtered = np.zeros((noisy_seq.shape[0], 4))

    b, a = butter(N=3, Wn=0.2, btype='low')

    step_1 = median_filter(noisy_seq[:, 0], size=5)
    step_2 = filtfilt(b, a, step_1)
    filtered[:, 0] = savgol_filter(step_2, window_length=15, polyorder=2)

    step_1 = median_filter(noisy_seq[:, 1], size=5)
    step_2 = filtfilt(b, a, step_1)
    filtered[:, 1] = savgol_filter(step_2, window_length=15, polyorder=2)

    step_1 = median_filter(noisy_seq[:, 3], size=5)
    step_2 = filtfilt(b, a, step_1)
    filtered[:, 2] = savgol_filter(step_2, window_length=51, polyorder=2)

    alpha_w = np.zeros(noisy_seq.shape[0])
    alpha_w[:-1] = (filtered[1:, 2] - filtered[:-1, 2]) / dt_val
    alpha_w[-1] = alpha_w[-2]

    filtered[:, 3] = savgol_filter(alpha_w, window_length=11, polyorder=2)

    return filtered

classical_train_np = np.array([apply_classical_pipeline(seq, dt) for seq in X_train_np])
classical_val_np = np.array([apply_classical_pipeline(seq, dt) for seq in X_val_np])

Classical_Train_Labels = torch.from_numpy(classical_train_np).to(device).float()
Classical_Val_Labels = torch.from_numpy(classical_val_np).to(device).float()

# Logika dynamicznego "Ground Truth"
if USE_REAL_DATA:
    Clean_Val_Target = Classical_Val_Labels[:, :, :3].clone()
else:
    clean_val_np = clean_all[num_train:]
    Clean_Val_Target = torch.from_numpy(clean_val_np).to(device).float()

def train_mamba_apprentice(name, target_labels):
    print(f"Training Mamba ({name})...")
    model = MambaLikeSSM(hidden_size=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    best_loss = float('inf')
    best_wts = copy.deepcopy(model.state_dict())
    patience, count = 60, 0

    for epoch in range(1500):
        model.train()
        optimizer.zero_grad()

        preds = model(Train_Input)
        loss = nn.MSELoss()(preds, target_labels)

        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out = model(Val_Input)
            v_loss = nn.MSELoss()(val_out[:, :, :3], Clean_Val_Target).item()

            if v_loss < best_loss:
                best_loss = v_loss
                best_wts = copy.deepcopy(model.state_dict())
                count = 0
            else:
                count += 1

        if count >= patience:
            break

    model.load_state_dict(best_wts)
    model.eval()
    print(f"Finished {name}. Best Val MSE on Ground Truth (States 0-2): {best_loss:.5f}\n")
    return model

mamba_classical = train_mamba_apprentice("Distilled: Classical Filter Pipeline", Classical_Train_Labels)
mamba_ekf = train_mamba_apprentice("Distilled: SINDy-EKF Labels", EKF_Train_Labels)

In [ ]:
# --- 8. INTERACTIVE VISUALIZATION (Dynamic Truth Handling) ---
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np

Val_Target_Noisy = torch.from_numpy(X_val_np).to(device).float()

def plot_interactive(seq_idx=0, start_idx=0, zoom_len=500, show_ekf=True, show_mamba_ekf=True, show_mamba_class=True):
    plt.figure(figsize=(16, 12))

    true_states = Clean_Val_Target[seq_idx].cpu().numpy()
    noisy = Val_Target_Noisy[seq_idx].cpu().numpy()
    u_vals = U_val_np[seq_idx]

    clean_theta = true_states[:, 0]
    clean_omega_p = true_states[:, 1]
    clean_omega_w = true_states[:, 2]

    # Obliczenie Alpha wprost z wektora Truth (wspiera zarówno symulację jak i filtr z danych zebranych)
    true_alpha_w = np.zeros_like(clean_omega_w)
    true_alpha_w[:-1] = (clean_omega_w[1:] - clean_omega_w[:-1]) / dt
    true_alpha_w[-1] = true_alpha_w[-2]

    true = np.column_stack((true_states, true_alpha_w))

    ekf_pred = EKF_Val_Labels[seq_idx].cpu().numpy()
    mamba_ekf_pred = mamba_ekf(Val_Input[seq_idx:seq_idx+1])[0].detach().cpu().numpy()
    mamba_class_pred = mamba_classical(Val_Input[seq_idx:seq_idx+1])[0].detach().cpu().numpy()

    t = np.arange(noisy.shape[0]) * dt

    end_idx = min(start_idx + zoom_len, noisy.shape[0])
    idx = slice(start_idx, end_idx)

    plots = [
        {"title": "1. Angle (rad)", "true_idx": 0, "meas_idx": 0, "state_idx": 0},
        {"title": "2. Pendulum Velocity - Gyro (rad/s)", "true_idx": 1, "meas_idx": 1, "state_idx": 1},
        {"title": "3. DC Motor Velocity - Encoder (rad/s)", "true_idx": 2, "meas_idx": 3, "state_idx": 2},
        {"title": "4. Flywheel Acceleration (rad/s^2)", "true_idx": 3, "meas_idx": None, "state_idx": 3}
    ]

    for i, p in enumerate(plots):
        plt.subplot(4, 1, i+1)

        if p["meas_idx"] is not None:
            plt.plot(t[idx], noisy[idx, p["meas_idx"]], color='gray', alpha=0.5, label='Hardware Measurement (Noisy)')
        else:
            raw_accel = np.gradient(noisy[:, 3], dt)
            plt.plot(t[idx], raw_accel[idx], color='gray', alpha=0.3, label='Raw Encoder Derivative')

        if p["true_idx"] is not None:
            label_text = 'Clean Truth (Unobservable)' if not USE_REAL_DATA else 'Best Filtered Estimate'
            plt.plot(t[idx], true[idx, p["true_idx"]], 'k--', linewidth=2, label=label_text)

        if show_ekf:
            plt.plot(t[idx], ekf_pred[idx, p["state_idx"]], color='orange', linewidth=2, label='SINDy-EKF Target')

        if show_mamba_class:
            plt.plot(t[idx], mamba_class_pred[idx, p["state_idx"]], color='green', linewidth=2, label='Mamba (Classical Trained)')

        if show_mamba_ekf:
            plt.plot(t[idx], mamba_ekf_pred[idx, p["state_idx"]], color='blue', linewidth=2, linestyle='--', label='Mamba (SINDy-EKF Trained)')

        plt.title(p["title"])
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude")
        plt.grid(True, alpha=0.3)

        if i == 3:
            plt.ylim(-150, 150)

        if i == 0: plt.legend(loc='upper right', ncol=3)

    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_interactive,
    seq_idx=widgets.IntSlider(min=0, max=len(Val_Target_Noisy)-1, step=1, value=0, description='Validation Seq:'),
    start_idx=widgets.IntSlider(min=0, max=4500, step=100, value=0, description='Start Time (idx):'),
    zoom_len=widgets.IntSlider(min=100, max=5000, step=100, value=500, description='Zoom Window:'),
    show_ekf=widgets.Checkbox(value=True, description='Show SINDy-EKF'),
    show_mamba_class=widgets.Checkbox(value=False, description='Show Mamba (Classical)'),
    show_mamba_ekf=widgets.Checkbox(value=False, description='Show Mamba (SINDy)')
);

In [ ]:
# --- 5. CLOSED-LOOP STATE FEEDBACK CONTROL WITH WHEEL DESATURATION & SINDy-EKF ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import numpy as np

print("Optimizing Full State Feedback Gains via advanced BPTT (with Nan-Guards)...")

# Parametry tylko dla trybu symulacji (USE_REAL_DATA = False)
m_p, l, g = 0.15, PENDULUM_LENGTH, GRAVITY
I_p, I_w = 0.003, 0.0001
b_p, b_w = 0.001, 0.0005
K_t, K_e, u_d = 0.05, 0.05, 1.5
INTEGRAL_DECAY = 0.9999

def physics_step_sim(theta, omega_p, omega_w, u, dt=0.01, n_substeps=10):
    # --- KWANTYZACJA STEROWANIA ---
    pwm_val = np.round((u / 12.0) * 255.0)
    pwm_val = np.clip(pwm_val, -255, 255)
    u_quantized = (pwm_val / 255.0) * 12.0

    # --- DYSKRETNOŚĆ CZASU (Sub-stepping) ---
    dt_sub = dt / n_substeps
    curr_theta, curr_omega_p, curr_omega_w = theta, omega_p, omega_w

    for _ in range(n_substeps):
        u_net = u_quantized - K_e * curr_omega_w
        u_eff = u_net - u_d * np.tanh(u_net / u_d)
        tau_m = K_t * u_eff

        alpha_p = (m_p * g * l * np.sin(curr_theta) - tau_m + b_w * curr_omega_w - b_p * curr_omega_p) / I_p
        alpha_w_new = (tau_m - b_w * curr_omega_w) / I_w - alpha_p

        curr_omega_p += alpha_p * dt_sub
        curr_theta += curr_omega_p * dt_sub
        curr_omega_w += alpha_w_new * dt_sub

    return curr_theta, curr_omega_p, curr_omega_w, alpha_p, alpha_w_new

# --- 1. OPTIMIZE GAIN AND COMPENSATORS (PyTorch - CONSTRAINED NONLINEARITIES) ---
print("Optimizing Full Gains with Strictly Constrained Friction & Back-EMF...")

C_tensor = torch.from_numpy(C_matrix).float()

def identified_physics_step(theta, omega_p, omega_w, u, C_tensor, dt_val):
    features = torch.stack([
        omega_p, omega_w, u,
        torch.sin(theta),
        torch.tanh(0.5 * u), torch.tanh(u),
        torch.tanh(2.0 * u), torch.tanh(5.0 * u)
    ], dim=0)

    x_dot = torch.matmul(C_tensor, features)
    next_theta = theta + x_dot[0] * dt_val
    next_omega_p = omega_p + x_dot[1] * dt_val
    next_omega_w = omega_w + x_dot[2] * dt_val
    return next_theta, next_omega_p, next_omega_w, x_dot[2]

K_pd = nn.Parameter(torch.tensor([-150.0, -20.0, 0.05, -10.0]))
w_comp_raw = nn.Parameter(torch.tensor([1.0]))
w_emf_raw = nn.Parameter(torch.tensor([0.05]))

optimizer = optim.AdamW([K_pd, w_comp_raw, w_emf_raw], lr=0.1, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=100, gamma=0.5)

max_steps = 400

for epoch in range(20):
    optimizer.zero_grad()
    loss = 0
    theta, omega_p, omega_w, alpha_w = torch.tensor([0.05]), torch.tensor([0.0]), torch.tensor([0.0]), torch.tensor([0.0])
    I_term = torch.tensor([0.0])
    u_delayed = torch.tensor([0.0])

    for step in range(max_steps):
        I_term = INTEGRAL_DECAY * I_term + theta * dt
        I_term = torch.clamp(I_term, -10.0, 10.0)

        K_angle_c = -torch.abs(K_pd[0])
        K_vel_c = -torch.abs(K_pd[1])
        K_wheel_c = K_pd[2]
        K_int_c = -torch.abs(K_pd[3])

        # --- FIZYCZNE LIMITY NA NIELINIOWOŚCI ---
        # Tarcie nie może przekroczyć 2.5 V (realistyczna strefa martwa silnika)
        w_frict_c = torch.clamp(torch.abs(w_comp_raw), 0.0, 3.0)
        # Stała Back-EMF nie może przekroczyć 0.2 V/(rad/s)
        w_emf_c = torch.clamp(torch.abs(w_emf_raw), 0.0, 0.2)

        u_raw = -(K_angle_c * theta + K_vel_c * omega_p + K_wheel_c * omega_w + K_int_c * I_term).squeeze()

        # Zastosowanie fizycznie zablokowanych wag
        u_total = u_raw + w_frict_c * torch.tanh(5.0 * u_raw) + w_emf_c * omega_w
        u = 12.0 * torch.tanh(u_total / 12.0)

        process_noise = torch.randn(1) * 0.001
        omega_p_noisy = omega_p + process_noise

        theta, omega_p, omega_w, alpha_w = identified_physics_step(theta, omega_p_noisy, omega_w, u_delayed, C_tensor, dt)
        u_delayed = u.clone()

        if torch.isnan(theta) or torch.isinf(theta) or torch.abs(theta) > 0.3:
            loss += (max_steps - step) * 10000.0
            break

        # Dodajemy karę za nadużywanie tarcia i EMF (zmusza to K do cięższej pracy)
        loss += (theta**2)*500.0 + (omega_p**2)*100.0 + (u_raw**2)*0.1 + (I_term**2)*0.001 + (omega_w**2)*0.01
        loss += (w_emf_c**2)*50000.0

    loss.backward()
    torch.nn.utils.clip_grad_norm_([K_pd, w_comp_raw, w_emf_raw], max_norm=0.5)

    grad_ok = True
    for p in [K_pd, w_comp_raw, w_emf_raw]:
        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
            grad_ok = False

    if grad_ok:
        optimizer.step()

    scheduler.step()

K_opt = np.array([
    -np.abs(K_pd.detach().numpy()[0]),
    -np.abs(K_pd.detach().numpy()[1]),
    K_pd.detach().numpy()[2],
    -np.abs(K_pd.detach().numpy()[3])
])
# Pobieramy realne, ograniczone wartości (clamp) aby zgadzały się z logiką
w_opt = np.clip(np.abs(w_comp_raw.detach().numpy()[0]), 0.0, 2.5)
w_emf_opt = np.clip(np.abs(w_emf_raw.detach().numpy()[0]), 0.0, 0.2)

print("\nOptimization Finished Safely!")
print(f"Optimized Full State Feedback Gain K: {K_opt}")
print(f"Learned Friction Kick: {w_opt:.4f} V")
print(f"Learned Back-EMF Feedforward: {w_emf_opt:.4f}")

# --- 2. CLOSED-LOOP SIMULATION SETUP ---
print("\nRunning Closed-Loop Simulation...")
_fast_physics_compiled = exec_globals['_fast_physics_compiled']

sim_steps = 30000
t_sim = np.arange(sim_steps) * dt
theta_bias = 0.05 * np.sin(0.5 * t_sim) + np.cumsum(np.random.normal(0, 0.001, size=sim_steps))

history_x, history_x_hat, history_u, history_z = np.zeros((sim_steps, 3)), np.zeros((sim_steps, 4)), np.zeros(sim_steps), np.zeros((sim_steps, 4))
curr_x, curr_x_hat = np.array([0.05, 0.0, 0.0]), np.array([0.05, 0.0, 0.0])
P_cov, curr_alpha_w = np.eye(3) * 0.1, 0.0
I_sim = 0.0

Q = np.array([[1e-5, 0.0, 0.0], [0.0, 1e-3, 0.0], [0.0, 0.0, 1e-3]])
R = np.array([[1e-2, 0, 0, 0], [0, 1e-2, 0, 0], [0, 0, 1e-1, 0], [0, 0, 0, 2.0]])
H = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [GRAVITY, 0.0, 0.0], [0.0, 0.0, 1.0]])

phi, phi_enc_prev, enc_res = 0.0, 0.0, (2 * np.pi) / PPR

for k in range(sim_steps):
    I_sim *= INTEGRAL_DECAY
    if abs(curr_x_hat[0]) > 0.01:
        I_sim += curr_x_hat[0] * dt
    I_sim = np.clip(I_sim, -10.0, 10.0)

    u_raw = -(K_opt[0] * curr_x_hat[0] + K_opt[1] * curr_x_hat[1] + K_opt[2] * curr_x_hat[2] + K_opt[3] * I_sim)
    u = np.clip(u_raw + w_opt * np.tanh(5.0 * u_raw) + w_emf_opt * curr_x_hat[2], -12.0, 12.0)
    history_u[k] = u

    if USE_REAL_DATA:
        pwm_val = np.round((u / 12.0) * 255.0)
        pwm_val = np.clip(pwm_val, -255, 255)
        u_quantized = (pwm_val / 255.0) * 12.0

        x_dot_true = _fast_physics_compiled(curr_x, float(u_quantized), C_matrix)
        curr_x = curr_x + x_dot_true * dt
        true_alpha_p, true_alpha_w = x_dot_true[1], x_dot_true[2]
    else:
        curr_x[0], curr_x[1], curr_x[2], true_alpha_p, true_alpha_w = physics_step_sim(curr_x[0], curr_x[1], curr_x[2], u, dt=dt)

    phi += curr_x[2] * dt
    history_x[k] = curr_x.copy()

    sens_theta = curr_x[0] + np.random.normal(0, 0.01) + theta_bias[k]
    sens_gyro = curr_x[1] + np.random.normal(0, 0.02)
    sens_accel = GRAVITY * np.sin(curr_x[0]) + PENDULUM_LENGTH * true_alpha_p + np.random.normal(0, 0.1)

    phi_enc = np.round(phi / enc_res) * enc_res
    sens_omega_w = (phi_enc - phi_enc_prev) / dt
    phi_enc_prev = phi_enc
    z_meas = np.array([sens_theta, sens_gyro, sens_accel, sens_omega_w])
    history_z[k] = z_meas

    x_dot = _fast_physics_compiled(curr_x_hat, float(u), C_matrix)
    x_pred = curr_x_hat + x_dot * dt

    eps_fd = 1e-4
    F_mat = np.zeros((3, 3))
    for i in range(3):
        x_plus = curr_x_hat.copy()
        x_plus[i] += eps_fd
        dot_plus = _fast_physics_compiled(x_plus, float(u), C_matrix)
        F_mat[:, i] = (dot_plus - x_dot) / eps_fd
    F_mat = np.eye(3) + F_mat * dt

    P_pred = F_mat @ P_cov @ F_mat.T + Q
    z_pred = H @ x_pred
    z_pred[2] = GRAVITY * np.sin(x_pred[0]) + PENDULUM_LENGTH * x_dot[1]

    S = H @ P_pred @ H.T + R
    K_ekf = P_pred @ H.T @ np.linalg.inv(S)

    curr_x_hat = x_pred + K_ekf @ (z_meas - z_pred)
    P_cov = (np.eye(3) - K_ekf @ H) @ P_pred

    curr_alpha_w = _fast_physics_compiled(curr_x_hat, float(u), C_matrix)[2]
    history_x_hat[k] = np.array([curr_x_hat[0], curr_x_hat[1], curr_x_hat[2], curr_alpha_w])

print("Simulation Complete.")

# --- AUTOMATYCZNE GENEROWANIE KODU DO ARDUINO ---
print("\n" + "="*60)
print("// --- KOPIUJ PONIŻSZY BLOK I WKLEJ GO DO ARDUINO IDE ---")
print("="*60)
print("// Indeksy: 0 = K_angle, 1 = K_velocity, 2 = K_wheel, 3 = K_integral")
print(f"volatile float K_gains[4] = {{{K_opt[0]:.8f}f, {K_opt[1]:.8f}f, {K_opt[2]:.8f}f, {K_opt[3]:.8f}f}};")
print(f"volatile float w_friction = {w_opt:.4f}f;")
print(f"volatile float w_emf = {w_emf_opt:.4f}f;\n")

print("const float C_matrix[3][8] = {")
for i, row in enumerate(C_matrix):
    row_str = ", ".join([f"{val:.4f}f" for val in row])
    comma = "," if i < 2 else ""
    print(f"  {{ {row_str} }}{comma}")
print("};")
print("="*60)

In [ ]:
# --- 2. CLOSED-LOOP SIMULATION SETUP ---
print("\nRunning Closed-Loop Simulation...")
_fast_physics_compiled = exec_globals['_fast_physics_compiled']

sim_steps = 500
t_sim = np.arange(sim_steps) * dt
theta_bias = 0.0 * np.sin(0.5 * t_sim) + np.cumsum(np.random.normal(0, 0.001, size=sim_steps))

history_x, history_x_hat, history_u, history_z = np.zeros((sim_steps, 3)), np.zeros((sim_steps, 4)), np.zeros(sim_steps), np.zeros((sim_steps, 4))
curr_x, curr_x_hat = np.array([0.01, 0.0, 0.0]), np.array([0.01, 0.0, 0.0])
P_cov, curr_alpha_w = np.eye(3) * 0.1, 0.0

Q = np.array([[1e-5, 0.0, 0.0], [0.0, 1e-3, 0.0], [0.0, 0.0, 1e-3]])
R = np.array([[1e-2, 0, 0, 0], [0, 1e-2, 0, 0], [0, 0, 1e-1, 0], [0, 0, 0, 2.0]])
H = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [GRAVITY, 0.0, 0.0], [0.0, 0.0, 1.0]])

phi, phi_enc_prev, enc_res = 0.0, 0.0, (2 * np.pi) / PPR

for k in range(sim_steps):
    state_fb = np.array([curr_x_hat[0], curr_x_hat[1], curr_x_hat[2], curr_alpha_w])
    u_raw = -np.dot(K_opt, state_fb)
    u = np.clip(u_raw + w_opt * np.tanh(5.0 * u_raw) + w_emf_opt * curr_x_hat[2], -12.0, 12.0)
    history_u[k] = u

    # --- KLUCZOWA ZMIANA: Ewolucja prawdziwego stanu w zależności od trybu ---
    if USE_REAL_DATA:
        x_dot_true = _fast_physics_compiled(curr_x, float(u), C_matrix)
        curr_x = curr_x + x_dot_true * dt
        true_alpha_p, true_alpha_w = x_dot_true[1], x_dot_true[2]
    else:
        curr_x[0], curr_x[1], curr_x[2], true_alpha_p, true_alpha_w = physics_step_sim(curr_x[0], curr_x[1], curr_x[2], u)

    phi += curr_x[2] * dt
    history_x[k] = curr_x.copy()

    sens_theta = curr_x[0] + np.random.normal(0, 0.01) + theta_bias[k]
    sens_gyro = curr_x[1] + np.random.normal(0, 0.02)
    sens_accel = GRAVITY * np.sin(curr_x[0]) + PENDULUM_LENGTH * true_alpha_p + np.random.normal(0, 0.1)

    phi_enc = np.round(phi / enc_res) * enc_res
    sens_omega_w = (phi_enc - phi_enc_prev) / dt
    phi_enc_prev = phi_enc
    z_meas = np.array([sens_theta, sens_gyro, sens_accel, sens_omega_w])
    history_z[k] = z_meas

    x_dot = _fast_physics_compiled(curr_x_hat, float(u), C_matrix)
    x_pred = curr_x_hat + x_dot * dt

    eps_fd = 1e-4
    F_mat = np.zeros((3, 3))
    for i in range(3):
        x_plus = curr_x_hat.copy()
        x_plus[i] += eps_fd
        dot_plus = _fast_physics_compiled(x_plus, float(u), C_matrix)
        F_mat[:, i] = (dot_plus - x_dot) / eps_fd
    F_mat = np.eye(3) + F_mat * dt

    P_pred = F_mat @ P_cov @ F_mat.T + Q
    z_pred = H @ x_pred
    z_pred[2] = GRAVITY * np.sin(x_pred[0]) + PENDULUM_LENGTH * x_dot[1]

    S = H @ P_pred @ H.T + R
    K_ekf = P_pred @ H.T @ np.linalg.inv(S)

    curr_x_hat = x_pred + K_ekf @ (z_meas - z_pred)
    P_cov = (np.eye(3) - K_ekf @ H) @ P_pred

    curr_alpha_w = _fast_physics_compiled(curr_x_hat, float(u), C_matrix)[2]
    history_x_hat[k] = np.array([curr_x_hat[0], curr_x_hat[1], curr_x_hat[2], curr_alpha_w])

print("Simulation Complete. Plotting Results...")

# --- 3. PLOTTING ---
plt.figure(figsize=(14, 10))
plt.subplot(3, 1, 1)
plt.plot(t_sim, history_x[:, 0], 'k-', linewidth=2, label='True Angle')
plt.plot(t_sim, history_z[:, 0], 'gray', alpha=0.5, label='Measured Angle (Bias)')
plt.plot(t_sim, theta_bias, 'r--', alpha=0.8, label='Injected Bias')
plt.plot(t_sim, history_x_hat[:, 0], 'b--', linewidth=2, label='EKF Estimated Angle')
plt.axhline(0, color='g', linestyle=':')
plt.ylabel('Angle (rad)')
plt.title('Reaction Wheel Pendulum closed-loop test on SINDy Equations')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 2)
plt.plot(t_sim, history_x[:, 1], 'k-', linewidth=2, label='True Pendulum Vel')
plt.plot(t_sim, history_x_hat[:, 1], 'b--', linewidth=2, label='EKF Pendulum Vel')
plt.ylabel('Velocity (rad/s)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(t_sim, history_x[:, 2], 'k-', linewidth=2, label='True Flywheel Vel')
plt.plot(t_sim, history_u, 'r-', linewidth=1.5, label='Compensated Voltage (U)')
plt.ylabel('Velocity / Volts')
plt.xlabel('Time (s)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()